## !!! Doesn't yet work fully for those data points with differing numbers of decimal points.

In [143]:
import pandas as pd
import numpy as np
from nocasedict import NocaseDict
import simplejson
import math
import json

pd.set_option('display.max_columns', None)

In [144]:
data1 = pd.read_csv('Raw Data Files//LGBF_ScotDataReal_ISSpatialHub.csv')
data2 = pd.read_csv('Raw Data Files//LGBF_ScotDataReal_ISSpatialHub_March.csv')

In [145]:
data1.head(3)

,Unnamed: 0,Indicators_Information_Code,LA_Information_LocalAuthority,LA_Data_LGBF_Year,LA_Data_LA_IndicatorReal,LA_Data_LA_Numerator_real,LA_Data_LA_Den_Real,Scotland_Data_Scotland_Indicator_Real,Scotland_Data_Scotland_Num_Real,Scotland_Data_Scotland_Den_Real,FG_Data_FG_Avg_Indicator_Real,FG_Data_FG_Avg_Num_Real,FG_Data_FG_Avg_Den_Real,FG_Data_FamilyGroup
0,0,C&L01,Aberdeen City,2010-11,0.4305,830.9841,1922292.0,4.6311,210605.5487,45459818.0,4.536500,11277.478237,2200418.875,Urban
1,1,C&L01,Aberdeen City,2011-12,0.9743,1992.1264,2045051.0,4.1406,199366.4679,48202343.0,4.406650,10768.731313,2355970.750,Urban
2,2,C&L01,Aberdeen City,2012-13,4.5821,9923.3499,2163756.0,3.9527,203940.3869,51624697.0,4.604163,11924.258775,2601296.500,Urban


In [146]:
data2.head(3)

,Column1,Code,LocalAuthority,Year,LA_IndicatorReal,LA_Numerator_real,LA_Den_Real,Scotland_Indicator_Real,Scotland_Num_Real,Scotland_Den_Real,FG_Avg_Indicator_Real,FG_Avg_Num_Real,FG_Avg_Den_Real,FamilyGroup
0,0,C&L01,Aberdeen City,2010-11,0.4305,830.9841,1922292.0,4.6311,210605.5487,45459818.0,4.536500,11277.47824,2200418.875,Urban
1,1,C&L01,Aberdeen City,2011-12,0.9743,1992.1264,2045051.0,4.1406,199366.4679,48202343.0,4.406650,10768.73131,2355970.750,Urban
2,2,C&L01,Aberdeen City,2012-13,4.5821,9923.3499,2163756.0,3.9527,203940.3869,51624697.0,4.604163,11924.25878,2601296.500,Urban


In [147]:
data1['Key'] = data1['Indicators_Information_Code'] + data1['LA_Information_LocalAuthority'] + data1['LA_Data_LGBF_Year']
data1 = data1[['Key','Indicators_Information_Code','LA_Information_LocalAuthority','LA_Data_LGBF_Year','LA_Data_LA_IndicatorReal']]
data1 = data1.rename(columns= {'Indicators_Information_Code': 'Code','LA_Information_LocalAuthority': 'LocalAuthority','LA_Data_LGBF_Year': 'Period', 'LA_Data_LA_IndicatorReal': 'Value'})
data1.head(3)

,Key,Code,LocalAuthority,Period,Value
0,C&L01Aberdeen City2010-11,C&L01,Aberdeen City,2010-11,0.4305
1,C&L01Aberdeen City2011-12,C&L01,Aberdeen City,2011-12,0.9743
2,C&L01Aberdeen City2012-13,C&L01,Aberdeen City,2012-13,4.5821


In [148]:
data2['Key'] = data2['Code'] + data2['LocalAuthority'] + data2['Year']
data2 = data2[['Key','Code','LocalAuthority','Year','LA_IndicatorReal']]
data2 = data2.rename(columns= {'Year': 'Period', 'LA_IndicatorReal': 'Value'})
data2.head(3)

,Key,Code,LocalAuthority,Period,Value
0,C&L01Aberdeen City2010-11,C&L01,Aberdeen City,2010-11,0.4305
1,C&L01Aberdeen City2011-12,C&L01,Aberdeen City,2011-12,0.9743
2,C&L01Aberdeen City2012-13,C&L01,Aberdeen City,2012-13,4.5821


In [149]:
data = data2.merge(data1[['Key','Value']],how='left', on='Key', suffixes=('_July', '_March'))
data.head(3)

,Key,Code,LocalAuthority,Period,Value_July,Value_March
0,C&L01Aberdeen City2010-11,C&L01,Aberdeen City,2010-11,0.4305,0.4305
1,C&L01Aberdeen City2011-12,C&L01,Aberdeen City,2011-12,0.9743,0.9743
2,C&L01Aberdeen City2012-13,C&L01,Aberdeen City,2012-13,4.5821,4.5821


In [150]:
info = pd.read_csv('Data Files//Indicator Information.csv')
data = data.merge(info[['Code','MeasureType']], how = 'left', on = 'Code')
data = data.query('MeasureType != "Cost"')
data = data.drop(columns=['MeasureType'])
data.head(3)

,Key,Code,LocalAuthority,Period,Value_July,Value_March
1621,C&L05aAberdeen City2010-14,C&L05a,Aberdeen City,2010-14,0.734,0.734
1622,C&L05aAberdeen City2012-15,C&L05a,Aberdeen City,2012-15,0.713,0.713
1623,C&L05aAberdeen City2013-16,C&L05a,Aberdeen City,2013-16,0.707,0.707


In [151]:
data = data.dropna(subset=["Value_March"])
data = data.round({'Value_July': 12, 'Value_March': 12})
def compare(df) :
    return df.Value_July == df.Value_March

data['Matches'] = data.apply(compare,axis = 1)
data = data.query('Matches == False')
data

,Key,Code,LocalAuthority,Period,Value_July,Value_March,Matches
4149,CHN04Aberdeen City2011-12,CHN04,Aberdeen City,2011-12,0.470000,0.500000,False
4150,CHN04Aberdeen City2012-13,CHN04,Aberdeen City,2012-13,0.480000,0.490000,False
4151,CHN04Aberdeen City2013-14,CHN04,Aberdeen City,2013-14,0.490000,0.520000,False
4152,CHN04Aberdeen City2014-15,CHN04,Aberdeen City,2014-15,0.520000,0.560000,False
4153,CHN04Aberdeen City2015-16,CHN04,Aberdeen City,2015-16,0.550000,0.590000,False
...,...,...,...,...,...,...,...
35461,SW08West Lothian2019-20,SW08,West Lothian,2019-20,934.422385,934.422385,False
35462,SW08West Lothian2020-21,SW08,West Lothian,2020-21,359.595340,359.595340,False
35463,SW08West Lothian2021-22,SW08,West Lothian,2021-22,426.368085,426.368085,False
35464,SW08West Lothian2022-23,SW08,West Lothian,2022-23,656.681002,656.681003,False


In [152]:
data.to_csv('test2.csv')